In [ ]:
!apt install fluidsynth

!git clone https://github.com/jthickstun/anticipation.git
!pip install ./anticipation
!pip install -r anticipation/requirements.txt

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fluid-soundfont-gm libevdev2 libfluidsynth3 libgudev-1.0-0 libinput-bin
  libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a libqt5dbus5
  libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libwacom-bin
  libwacom-common libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1
  libxcb-render-util0 libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1
  libxkbcommon-x11-0 qsynth qt5-gtk-platformtheme qttranslations5-l10n
  timgm6mb-soundfont
Suggested packages:
  fluid-soundfont-gs qt5-image-formats-plugins qtwayland5 jackd
The following NEW packages will be installed:
  fluid-soundfont-gm fluidsynth libevdev2 libfluidsynth3 libgudev-1.0-0
  libinput-bin libinput10 libinstpatch-1.0-2 libmd4c0 libmtdev1 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libwacom-bin
  libwacom-common libwacom9 libx

In [ ]:
import sys,time
import numpy as np

import midi2audio
import transformers
import os

import torch
import torch.nn.functional as F

import random

from transformers import AutoModelForCausalLM
from transformers import BertConfig, BertModel
from pathlib import Path
from IPython.display import Audio

from anticipation import ops
from anticipation.sample import generate
from anticipation.tokenize import extract_instruments
from anticipation.convert import events_to_midi,midi_to_events
from anticipation.config import *
from anticipation.vocab import *


os.environ['TORCH_USE_CUDA_DSA'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [ ]:
import importlib
import anticipation.sample as sample
importlib.reload(sample)

<module 'anticipation.sample' from '/usr/local/lib/python3.11/dist-packages/anticipation/sample.py'>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
''' load our model '''

import torch
from transformers import BertConfig, BertModel

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# rebuild configuration
configuration = BertConfig()
configuration.vocab_size = 55128
configuration.max_position_embeddings = 2048

# load model weights
structure_derivation_model = BertModel(configuration).to(device)

# load the saved state
checkpoint_path = "/content/drive/MyDrive/MusicData/bert_checkpoints/structure_derivation_model.pth"
structure_derivation_model.load_state_dict(torch.load(checkpoint_path, map_location=device))

# set to evaluation mode
structure_derivation_model.eval()

Using device: cuda


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(55128, 768, padding_idx=0)
    (position_embeddings): Embedding(2048, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
 

In [ ]:
''' load each AMT model configuration '''

SMALL_MODEL = 'stanford-crfm/music-small-800k'     # faster inference, worse sample quality
MEDIUM_MODEL = 'stanford-crfm/music-medium-800k'   # slower inference, better sample quality
LARGE_MODEL = 'stanford-crfm/music-large-800k'     # slowest inference, best sample quality

# Load AMT models
AMT_S = AutoModelForCausalLM.from_pretrained(SMALL_MODEL).cuda()
AMT_M = AutoModelForCausalLM.from_pretrained(MEDIUM_MODEL).cuda()
AMT_L = AutoModelForCausalLM.from_pretrained(LARGE_MODEL).cuda()

# a MIDI synthesizer
fs = midi2audio.FluidSynth('/usr/share/sounds/sf2/FluidR3_GM.sf2')

# the MIDI synthesis script
def synthesize(fs, tokens):
    mid = events_to_midi(tokens)
    mid.save('tmp.mid')
    fs.midi_to_audio('tmp.mid', 'tmp.wav')
    return 'tmp.wav'

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at stanford-crfm/music-small-800k were not used when initializing GPT2LMHeadModel: ['token_out_embeddings']
- This IS expected if you are initializing GPT2LMHeadM

In [ ]:
''' load lakh midi dataset '''

import kagglehub
from sklearn.model_selection import train_test_split
from pathlib import Path

# download latest version of clean lakh dataset
path = kagglehub.dataset_download("imsparsh/lakh-midi-clean")
midi_paths=Path(path)
print("Path to dataset files:", midi_paths)

# list all .mid files
midi_files = list(midi_paths.rglob("*.mid"))

# read all the cleaned filepaths
path = '/content/drive/MyDrive/MusicData/clean_midi_files_second.txt'
lm_midi_files = []
with open(path, 'r') as f:
    for line in f:
        lm_midi_files.append(Path(line.strip()))

print (lm_midi_files[1])

Path to dataset files: /kaggle/input/lakh-midi-clean
/kaggle/input/lakh-midi-clean/Sparks/When_I_Kiss_You.mid


In [ ]:
''' load symphonynet dataset '''

from pathlib import Path

# access symphonynet data
sn_path = Path('/content/drive/MyDrive/MusicData/SymphonyNet_Dataset/contemporary')
sn_midi_files = list(sn_path.rglob("*.mid"))

print(sn_midi_files[1])

/content/drive/MyDrive/MusicData/SymphonyNet_Dataset/contemporary/29789.mid


In [ ]:
''' create dataset '''
ds = []
files = lm_midi_files + sn_midi_files
for item in files:
  ds.append(str(item))


print (ds[0])

/kaggle/input/lakh-midi-clean/Sparks/When_Do_I_Get_to_Sing_My_Way.mid


In [ ]:
''' set config. for experiments '''

TEMPERATURES   = [1.5,1.7, 2]        # various temperatures for testing
TOP_P = 0.98
N_PROMPTS      = 15                     # number of prompts you want to sample
GEN_START_SEC  = 10                     # start time for generation
GEN_END_SEC    = 60                     # end time (about 50 seconds of generation)
SEGMENT_TOTAL_TIME = 60
INDIVIDUAL_SEGMENT_LENGTHS = 10
MAX_TOKENS = 2048
CSV_PATH = '/content/drive/MyDrive/MusicData/temperature_eval_results.csv'

MODELS = {
    "AMT-S": AMT_S,
    "AMT-M": AMT_M,
    "AMT-L": AMT_L,
}

In [ ]:
''' create necessary functions '''

import os, json, time, random
from datetime import datetime
import pandas as pd

# function for extracting 10-sec segment for generation prompt
def extraction(ds):
  prompt = []

  # load a random file from the dataset
  file = random.sample(ds,1)
  # tokenise file
  events = midi_to_events(file[0])
  # extract first 10-seconds from the file
  segment = ops.clip(events, 0, 10)
  # use this as the prompt for generation
  prompt.append(segment)

  return prompt

# function to get prompt
def get_prompt():
    prompt = extraction(ds)
    return prompt.copy()

# function to pass embeddings through model & retrieve similarity scores
@torch.no_grad()
def mean_anchor_similarity(token_ids_list: list[torch.Tensor]) -> float:
    embeddings = []
    # for each segment in the token id's list
    for seg in token_ids_list:
        # move segment to correct device
        seg = seg.to(device)
        # pass segment through SD model
        out = structure_derivation_model(input_ids=seg)
        # extract embeddings
        emb = out.last_hidden_state[:, 0]
        # normalise outpits
        emb = F.normalize(emb, dim=-1)
        embeddings.append(emb)
    # define anchor segment & subsequent segments
    anchor = embeddings[0]
    others = embeddings[1:]
    # calculate similarity between each anchor&other segment pairs
    scores = [F.cosine_similarity(anchor, o, dim=-1).item() for o in others]
    # return similarity
    return float(np.mean(scores)) if scores else float("nan")

# function to segment a generated piece into 10-second chunks
def segment(sequence, total_duration, segment_length):
  tokens_per_second = len(sequence) / total_duration
  tokens_per_segment = int(tokens_per_second * segment_length)

  segments = []
  for i in range(0, len(sequence), tokens_per_segment):
      segment = sequence[i:i + tokens_per_segment]
      if len(segment) == tokens_per_segment:
          segments.append(segment)

  return segments

# evaluatation function (input: temperature)
def evaluate(amt_model, temperature: float) -> float:
    # build prompt
    history = get_prompt()

    # generate piece
    generated = sample.generate(
        amt_model,
        start_time=GEN_START_SEC,
        end_time=GEN_END_SEC,
        inputs=history,
        top_p=TOP_P,
        temperature=temperature,
        debug=False
    )
    # tokenise & segment
    mid = events_to_midi(generated)
    tokenised = midi_to_events(mid)
    segments = segment(tokenised, SEGMENT_TOTAL_TIME, INDIVIDUAL_SEGMENT_LENGTHS)

    print(f"[debug] T={temperature} | gen_len={len(generated)} | tok_len={len(tokenised)} | segs={len(segments)}")

    # filter out samples that are None or empty
    if not segments:
      print ("no segments")

    # skip if too few segments
    if len(segments) < 2:
      print ("too little segments")

    # truncate each segment if necessary
    truncated_segments = [s[:MAX_TOKENS] for s in segments]



    # prepare tensors
    input_ids_list = [torch.tensor(s, dtype=torch.long).unsqueeze(0).to(device) for s in segments]

    # final check, extract any invalid tokens
    final_input_ids_list = [s for s in input_ids_list if not ((s >= 55028).any() or (s < 0).any())]

    # compute mean similarity
    return mean_anchor_similarity(final_input_ids_list)

# function to run across different models & temperatures
def run_experiments(structure_derivation_model,n_prompts: int = N_PROMPTS):
    rows = []

    # for each model
    for model_name, model_obj in MODELS.items():
        # put model into eval mode
        structure_derivation_model.eval()
        # for each temperature, calculate a mean SD similarity score across 30 prompts
        for T in TEMPERATURES:
            scores = []
            for i in range(n_prompts):
                try:
                    score = evaluate(model_obj, temperature=T)
                # if error, report
                except Exception as e:
                    print(f"[WARN] {model_name} T={T} prompt={i+1}/{n_prompts} failed: {e}")
                    score = float("nan")
                scores.append(score)

            # store the scores
            scores_arr = np.array(scores, dtype=np.float32)
            mean_score = np.nanmean(scores_arr)
            std_score  = np.nanstd(scores_arr)
            valid_n    = int(np.sum(~np.isnan(scores_arr)))

            rows.append({
                "model": model_name,
                "temperature": T,
                "top_p": TOP_P,
                "n_prompts": n_prompts,
                "valid": valid_n,
                "mean_similarity": float(mean_score),
                "std_similarity": float(std_score),
                "scores": [float(x) if np.isfinite(x) else None for x in scores]
            })

            print(f"[{model_name}] T={T:.2f} | prompts={valid_n}/{n_prompts} | mean={mean_score:.4f} ± {std_score:.4f}")

    # save data in df
    df = pd.DataFrame([{
        "model": r["model"],
        "temperature": r["temperature"],
        "top_p": r["top_p"],
        "n_prompts": r["n_prompts"],
        "valid": r["valid"],
        "mean_similarity": r["mean_similarity"],
        "std_similarity": r["std_similarity"],
    } for r in rows])

    if os.path.exists(CSV_PATH):
        # append without duplicating header
        df.to_csv(CSV_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(CSV_PATH, index=False)

    # also write per-prompt scores to JSONL
    jsonl_path = os.path.splitext(CSV_PATH)[0] + "_scores.jsonl"
    with open(jsonl_path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")





In [ ]:
''' run '''

run_experiments(structure_derivation_model)

5096it [00:08, 574.29it/s]


[debug] T=1.5 | gen_len=339 | tok_len=339 | segs=6


5072it [00:09, 540.12it/s]


[debug] T=1.5 | gen_len=366 | tok_len=366 | segs=6


5065it [00:11, 431.41it/s]


...

[debug] T=2 | gen_len=33 | tok_len=33 | segs=6


 86%|████████▌ | 4301/5000 [00:14<00:02, 303.60it/s] 


[debug] T=2 | gen_len=195 | tok_len=195 | segs=6


  2%|▏         | 100/5000 [00:00<00:23, 208.08it/s]

[debug] T=2 | gen_len=6 | tok_len=6 | segs=6
[AMT-L] T=2.00 | prompts=14/15 | mean=0.5924 ± 0.1504


In [ ]:
import pandas as pd

with open('/content/drive/MyDrive/MusicData/temperature_eval_results_scores.jsonl', 'r') as json_file:
    json_list = list(json_file)

for json_str in json_list:
    result = json.loads(json_str)
    print(f"result: {result}")
    print(isinstance(result, dict))

result: {'model': 'AMT-S', 'temperature': 1.5, 'top_p': 0.98, 'n_prompts': 15, 'valid': 15, 'mean_similarity': 0.7558215856552124, 'std_similarity': 0.03258838877081871, 'scores': [0.6778501272201538, 0.7446326851844788, 0.8100218892097473, 0.7320910453796386, 0.7984025120735169, 0.7584500908851624, 0.7913300156593323, 0.7769687056541443, 0.7288094520568847, 0.7436904311180115, 0.7323166847229003, 0.7406813263893127, 0.7693109273910522, 0.7841712951660156, 0.7485960841178894]}
True
result: {'model': 'AMT-S', 'temperature': 1.7, 'top_p': 0.98, 'n_prompts': 15, 'valid': 14, 'mean_similarity': 0.7392087578773499, 'std_similarity': 0.03807658702135086, 'scores': [0.706026840209961, 0.7692974090576172, 0.6641169905662536, 0.7786206245422364, 0.6777572512626648, 0.7660822153091431, 0.7840545415878296, 0.7469920516014099, 0.7182541489601135, 0.7563776016235352, 0.7377338528633117, 0.7018478751182556, 0.7568993449211121, None, 0.7848608016967773]}
True
result: {'model': 'AMT-S', 'temperature':